In [1]:
import os
import numpy as np
import SimpleITK as sitk
import concurrent.futures
from radiomics import featureextractor

# Configuración de extractores por modalidad
extractors = {
    "t2w": featureextractor.RadiomicsFeatureExtractor("get_binWidth_Params/Params_T2w.yaml"),
    "hbv": featureextractor.RadiomicsFeatureExtractor("get_binWidth_Params/Params_DWI.yaml"),
    "adc": featureextractor.RadiomicsFeatureExtractor("get_binWidth_Params/Params_ADC.yaml")
}

# Configuración para que cada extractor solo calcule la característica Range del grupo firstorder
for modality, extractor in extractors.items():
    extractor.disableAllFeatures()
    extractor.enableFeaturesByName(firstorder=["Range"])

# Diccionario para almacenar los rangos para cada modalidad
modality_ranges = {
    "t2w": [],
    "hbv": [],
    "adc": []
}

def process_file(image_path, modality, extractor):
    """
    Procesa una imagen dada su ruta, modalidad y extractor correspondiente.
    Retorna una tupla (modalidad, range_value) o (modalidad, None) en caso de error.
    """
    try:
        image = sitk.ReadImage(image_path)
        # Crear una máscara completa para cubrir todo el volumen
        image_array = sitk.GetArrayFromImage(image)
        mask_array = np.ones_like(image_array, dtype=np.uint8)
        mask = sitk.GetImageFromArray(mask_array)
        mask.CopyInformation(image)
        
        # Extraer la característica
        result = extractor.execute(image, mask)
        if "original_firstorder_Range" in result:
            range_value = result["original_firstorder_Range"]
            return modality, range_value
        else:
            print(f"Imagen: {image_path} no contiene 'original_firstorder_Range' en el resultado para {modality}.")
            return modality, None
    except Exception as e:
        print(f"Error procesando {image_path} para {modality}: {e}")
        return modality, None

In [ ]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
import concurrent.futures
from radiomics import featureextractor

# Path to your CSV file containing image paths
CSV_PATH = '/home/jaalzate/Radiomics-Prostate-Cancer/artifacts/bimcv_data/BIMCV_with_predictions_filtered_volume.csv'

# Configuración de extractores por modalidad	
extractors = {
    'image_t2': featureextractor.RadiomicsFeatureExtractor('/home/jaalzate/Radiomics-Prostate-Cancer/data_analysis/z_get_binWidth/get_binWidth_Params/Params_T2w.yaml'),
    'image_dwi': featureextractor.RadiomicsFeatureExtractor('/home/jaalzate/Radiomics-Prostate-Cancer/data_analysis/z_get_binWidth/get_binWidth_Params/Params_DWI.yaml'),
    'image_adc': featureextractor.RadiomicsFeatureExtractor('/home/jaalzate/Radiomics-Prostate-Cancer/data_analysis/z_get_binWidth/get_binWidth_Params/Params_ADC.yaml')
}

# Configurar cada extractor para calcular solo la característica "Range" del grupo firstorder
for modality, extractor in extractors.items():
    extractor.disableAllFeatures()
    extractor.enableFeaturesByName(firstorder=['Range'])

# Diccionario para almacenar los valores de Range según modalidad\mmodality_ranges = {mod: [] for mod in extractors}


def process_file(image_path, modality, extractor):
    """
    Procesa una imagen: lee el fichero, genera máscara completa, extrae la característica Range.
    Retorna (modality, range_value) o (modality, None) en caso de error.
    """
    try:
        image = sitk.ReadImage(image_path)
        image_array = sitk.GetArrayFromImage(image)
        mask_array = np.ones_like(image_array, dtype=np.uint8)
        mask_array[0, 0, 0] = 0  # Así hay fondo y ROI
        mask = sitk.GetImageFromArray(mask_array)
        mask.CopyInformation(image)

        result = extractor.execute(image, mask)
        key = 'original_firstorder_Range'
        if key in result:
            return modality, result[key]
        else:
            print(f"Advertencia: '{key}' no encontrado en resultado de {modality} para {image_path}")
            return modality, None

    except Exception as e:
        print(f"Error procesando {modality} en {image_path}: {e}")
        return modality, None



df = pd.read_csv(CSV_PATH)

    # Preparar tareas: una tupla (ruta, modalidad, extractor)
tasks = []
for modality, extractor in extractors.items():
    if modality not in df.columns:
        raise ValueError(f"La columna '{modality}' no existe en el CSV")
    for img_path in df[modality].dropna():
        tasks.append((img_path, modality, extractor))

# Procesamiento concurrente\print(f"Procesando {len(tasks)} imágenes en paralelo...")
with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    futures = [executor.submit(process_file, path, mod, ext) for path, mod, ext in tasks]
    for f in concurrent.futures.as_completed(futures):
        mod, val = f.result()
        modality_ranges[mod].append(val)

# Mostrar resultados
for mod, values in modality_ranges.items():
    print(f"\nModalidad: {mod}")
    print(f"  Valores de Range extraídos ({len(values)}): {values}")

# Opcional: guardar resultados en un CSV
out_df = pd.DataFrame({
    'modality': [m for m in modality_ranges for _ in modality_ranges[m]],
    'range': [v for values in modality_ranges.values() for v in values]
})
out_df.to_csv('radiomics_ranges_results.csv', index=False)
print("Resultados guardados en 'radiomics_ranges_results.csv'")


In [ ]:
# Directorio base que contiene las carpetas de pacientes
base_dir = "../../../data/images"

# Recolectamos las tareas a ejecutar en paralelo
tasks = []
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = []
    # Recorremos cada carpeta de paciente
    for patient_id in os.listdir(base_dir):
        patient_path = os.path.join(base_dir, patient_id)
        if not os.path.isdir(patient_path):
            continue
        # Buscamos archivos que sean imágenes (terminadas en .mha)
        for file_name in os.listdir(patient_path):
            if not file_name.lower().endswith(".mha"):
                continue
            image_path = os.path.join(patient_path, file_name)
            # Determinar la modalidad en base al nombre del archivo
            for modality, extractor in extractors.items():
                if modality in file_name.lower():
                    futures.append(executor.submit(process_file, image_path, modality, extractor))
                    break

    # Recoger los resultados
    for future in concurrent.futures.as_completed(futures):
        modality, value = future.result()
        if value is not None:
            modality_ranges[modality].append(value)

In [3]:
# Mostrar estadísticas y sugerencias para cada modalidad
for modality, ranges in modality_ranges.items():
    print(f"\nModalidad: {modality.upper()}")
    if ranges:
        ranges_array = np.array(ranges)
        mean_range = np.mean(ranges_array)
        min_range = np.min(ranges_array)
        max_range = np.max(ranges_array)
        print("Estadísticas de firstorder:Range:")
        print(f"  Media: {mean_range:.2f}")
        print(f"  Mínimo: {min_range:.2f}")
        print(f"  Máximo: {max_range:.2f}")

        # Sugerir un binWidth basado en la media del rango y distintos números objetivo de bins
        target_bins_list = [16, 32, 64, 128]
        print("Sugerencia de binWidth (usando la media del firstorder:Range):")
        for tb in target_bins_list:
            suggested_bw = mean_range / tb
            print(f"  Para {tb} bins: binWidth = {suggested_bw:.2f}")
    else:
        print("No se encontraron imágenes o no se pudo extraer 'firstorder:Range'.")


Modalidad: T2W
Estadísticas de firstorder:Range:
  Media: 763.80
  Mínimo: 483.25
  Máximo: 3162.15
Sugerencia de binWidth (usando la media del firstorder:Range):
  Para 16 bins: binWidth = 47.74
  Para 32 bins: binWidth = 23.87
  Para 64 bins: binWidth = 11.93
  Para 128 bins: binWidth = 5.97

Modalidad: HBV
Estadísticas de firstorder:Range:
  Media: 2370.89
  Mínimo: 587.69
  Máximo: 11575.83
Sugerencia de binWidth (usando la media del firstorder:Range):
  Para 16 bins: binWidth = 148.18
  Para 32 bins: binWidth = 74.09
  Para 64 bins: binWidth = 37.05
  Para 128 bins: binWidth = 18.52

Modalidad: ADC
Estadísticas de firstorder:Range:
  Media: 3921.11
  Mínimo: 2033.00
  Máximo: 5457.00
Sugerencia de binWidth (usando la media del firstorder:Range):
  Para 16 bins: binWidth = 245.07
  Para 32 bins: binWidth = 122.53
  Para 64 bins: binWidth = 61.27
  Para 128 bins: binWidth = 30.63
